# Database Normalization, Functional Dependencies, and Relational Calculus

**Syllabus mapping:** relational model: relational algebra, tuple calculus;
integrity constraints; normal form.

**Objectives:** compute attribute closures $X^+$; find all candidate keys;
test for 2NF, 3NF, and BCNF violations; evaluate lossless join decomposition;
formulate and understand declarative Tuple Relational Calculus (TRC) queries.

## Theoretical Foundations

### 1. Functional Dependencies and Keys
- **Attribute Closure $(X)^+$:** The set of all attributes functionally determined by $X$ under FD set $F$.
- **Superkey:** $K$ is a superkey if $(K)^+ = R$.
- **Candidate Key:** A minimal superkey (no proper subset of $K$ is a superkey).
- **Prime Attribute:** An attribute that is a member of *any* candidate key.

### 2. Normal Forms Hierarchy
$$BCNF \subset 3NF \subset 2NF \subset 1NF$$
- **1NF:** All attribute domains contain only atomic (indivisible) values.
- **2NF:** 1NF and **no partial dependency** (no non-prime attribute depends on a proper subset of any candidate key).
- **3NF:** For every non-trivial FD $X \to Y$:
  - $X$ is a superkey, **OR**
  - $Y$ is a prime attribute.
- **BCNF:** For every non-trivial FD $X \to Y$:
  - $X$ **must** be a superkey.

### 3. Lossless Join Decomposition
A decomposition of $R$ into $(R_1, R_2)$ is **lossless** if and only if:
$$(R_1 \cap R_2) \to R_1 \in F^+ \quad \text{or} \quad (R_1 \cap R_2) \to R_2 \in F^+$$

### 4. Tuple Relational Calculus (TRC)
Declarative query language: $\{t \mid P(t)\}$.
- Existential quantifier: $\exists s \in R (P(s))$
- Universal quantifier: $\forall s \in R (P(s))$

In [ ]:
from itertools import combinations

# Attribute Closure Algorithm
def compute_closure(attributes, fds):
    closure = set(attributes)
    while True:
        updated = False
        for lhs, rhs in fds:
            if lhs.issubset(closure) and not rhs.issubset(closure):
                closure.update(rhs)
                updated = True
        if not updated:
            break
    return closure

# Candidate Keys Finder
def find_candidate_keys(relation_attrs, fds):
    all_attrs = set(relation_attrs)
    candidate_keys = []
    for r in range(1, len(all_attrs) + 1):
        for subset in combinations(sorted(all_attrs), r):
            s = set(subset)
            # Minimal check: no existing CK is a subset of s
            if any(set(ck).issubset(s) for ck in candidate_keys):
                continue
            if compute_closure(s, fds) == all_attrs:
                candidate_keys.append("".join(sorted(s)))
    return candidate_keys

# Test Relation R(A, B, C, D, E)
R_attrs = set("ABCDE")
F = [
    ({"A"}, {"B", "C"}),
    ({"C", "D"}, {"E"}),
    ({"B"}, {"D"}),
    ({"E"}, {"A"}),
]

keys = find_candidate_keys(R_attrs, F)
print(f"Relation R(A, B, C, D, E)")
print(f"Candidate Keys: {keys} (Total: {len(keys)})")

prime_attrs = set("".join(keys))
non_prime = R_attrs - prime_attrs
print(f"Prime Attributes: {sorted(prime_attrs)}")
print(f"Non-Prime Attributes: {sorted(non_prime)}")

# Check Normal Forms
def check_normal_forms(relation_attrs, fds, keys):
    all_attrs = set(relation_attrs)
    prime = set("".join(keys))
    key_sets = [set(k) for k in keys]

    is_bcnf, is_3nf, is_2nf = True, True, True
    for lhs, rhs in fds:
        non_trivial_rhs = rhs - lhs
        if not non_trivial_rhs:
            continue
        lhs_is_superkey = compute_closure(lhs, fds) == all_attrs
        rhs_is_prime = non_trivial_rhs.issubset(prime)

        if not lhs_is_superkey:
            is_bcnf = False
            if not rhs_is_prime:
                is_3nf = False
            # 2NF check: partial dependency
            for k in key_sets:
                if lhs.issubset(k) and lhs != k and not non_trivial_rhs.issubset(prime):
                    is_2nf = False

    return {"2NF": is_2nf, "3NF": is_3nf, "BCNF": is_bcnf}

nf_status = check_normal_forms(R_attrs, F, keys)
print("Normal Form Status:", nf_status)

# Lossless Join Test for Decomposition R1(A, B, C), R2(C, D, E)
R1, R2 = set("ABC"), set("CDE")
common = R1.intersection(R2)
closure_common = compute_closure(common, F)
is_lossless = R1.issubset(closure_common) or R2.issubset(closure_common)
print(f"Decomposition (ABC, CDE) Common: {common}, Closure: {closure_common}")
print(f"Is Lossless Join? {is_lossless}")

## GATE-Style Practice

**NAT:** Consider a relation schema $R(A, B, C, D, E)$ with functional dependencies
$F = \{A \to BC, CD \to E, B \to D, E \to A\}$. What is the total number
of **candidate keys** for relation $R$?

**MCQ:** Which of the following normal form decomposition guarantees is true?

A. Decomposition into 3NF is always lossless and dependency preserving; decomposition into BCNF is always lossless but may not preserve dependencies.
B. Decomposition into BCNF is always dependency preserving, but 3NF is not.
C. Every relation in 3NF is also in BCNF.
D. A relation with only two attributes can never be in BCNF.

**MSQ:** Let relation $R(A, B, C, D)$ satisfy functional dependencies
$F = \{AB \to C, C \to D, D \to A\}$. Which of the following statements are TRUE?

A. The candidate keys of $R$ are $AB, BC,$ and $BD$.
B. Attribute $A$ is a prime attribute.
C. The relation $R$ is in 3NF.
D. The relation $R$ is in BCNF.

## Solutions

NAT: **4**.
The candidate keys are **$A, E, CD,$ and $BC$** (Total: 4).
- $A^+ = \{A, B, C, D, E\} \implies A$ is a CK.
- $E \to A \implies E^+ = \{A, B, C, D, E\} \implies E$ is a CK.
- $(CD)^+ = \{C, D, E, A, B\} \implies CD$ is a CK (minimal since $C^+ = \{C\}, D^+ = \{D\}$).
- $(BC)^+$: $B \to D \implies BC \to CD \to E \to A \implies BC$ is a CK.

MCQ: **A**. 3NF decomposition can always achieve both lossless join and dependency
preservation simultaneously (via synthesis algorithm). BCNF decomposition guarantees
lossless join, but some dependencies may be lost.

MSQ: **A, B, C**.
- Candidate keys:
  - $(AB)^+ = \{A, B, C, D\} \implies AB$ is a CK.
  - $(BC)^+ = \{B, C, D, A\} \implies BC$ is a CK.
  - $(BD)^+ = \{B, D, A, C\} \implies BD$ is a CK.
  So A is true.
- Prime attributes are all members of candidate keys: $\{A, B, C, D\}$. All attributes are prime!
  Since all attributes are prime, every FD $X \to Y$ has prime RHS, satisfying the 3NF condition.
  So B and C are true.
- D is false: In $C \to D$, $C$ is not a superkey, violating BCNF.